In [1]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import randint
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

from config import SEED

Elegimos dataset

In [2]:
dataset = 'aug_sin_proc'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15972, 64128), (15972,), (3996, 64128), (3996,))

### Random Forest

#### Entrenamiento

In [8]:
modelo_ciclos = joblib.load(f'./modelos_clasicos/modelos/ciclos/{dataset}/melspec_rf.pkl')

In [9]:
modelo_ciclos

,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,20
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': [15, 20, 25],
    'min_samples_split': [15, 20, 25],
    'min_samples_leaf': [5, 10 , 15]
}

In [11]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=15, min_samples_leaf=15, min_samples_split=25;, score=0.644 total time= 2.8min
[CV 2/5] END max_depth=15, min_samples_leaf=15, min_samples_split=25;, score=0.664 total time= 2.5min
[CV 3/5] END max_depth=15, min_samples_leaf=15, min_samples_split=25;, score=0.645 total time= 2.6min
[CV 4/5] END max_depth=15, min_samples_leaf=15, min_samples_split=25;, score=0.665 total time= 2.7min
[CV 5/5] END max_depth=15, min_samples_leaf=15, min_samples_split=25;, score=0.668 total time= 2.7min
[CV 1/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.651 total time= 3.2min
[CV 2/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.662 total time= 3.1min
[CV 3/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.654 total time= 3.1min
[CV 4/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.666 total time= 3.0min
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [15, 20, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [15, 20, ...]}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [12]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,20,5,15,0.665726
3,25,10,15,0.663351
8,20,10,15,0.661828
1,20,10,20,0.661828
4,15,5,15,0.661727
7,20,15,25,0.661494
6,20,15,20,0.661494
5,20,5,25,0.659992
9,25,15,15,0.659473
0,15,15,25,0.657167


In [13]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 15, 'min_samples_leaf': 5, 'max_depth': 20}
Best CV score: 0.6657256513330256


In [14]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7732
           1       1.00      1.00      1.00      8240

    accuracy                           1.00     15972
   macro avg       1.00      1.00      1.00     15972
weighted avg       1.00      1.00      1.00     15972



#### Evaluación

In [15]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.59      0.47      0.52      1936
           1       0.58      0.69      0.63      2060

    accuracy                           0.58      3996
   macro avg       0.58      0.58      0.58      3996
weighted avg       0.58      0.58      0.58      3996



#### Guardado

In [16]:
os.makedirs(f'./modelos_clasicos/modelos/ventanas/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/ventanas/aug_sin_proc/melspec_rf.pkl']

## Features de Audio

In [3]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((15972, 46), (15972,), (3996, 46), (3996,))

### Random Forest

#### Entrenamiento

In [5]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': randint(15, 25),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(3, 10)
}

In [7]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.672 total time=  11.2s
[CV 2/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.678 total time=  13.4s
[CV 3/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.681 total time=  12.2s
[CV 4/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.687 total time=  13.4s
[CV 5/5] END max_depth=21, min_samples_leaf=6, min_samples_split=27;, score=0.692 total time=  12.8s
[CV 1/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.674 total time=  13.7s
[CV 2/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.677 total time=  12.3s
[CV 3/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.684 total time=  12.6s
[CV 4/5] END max_depth=22, min_samples_leaf=7, min_samples_split=19;, score=0.689 total time=  12.6s
[CV 5/5] END max_depth=22, min

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....00239E55006E0>, 'min_samples_leaf': <scipy.stats....00239B48239D0>, 'min_samples_split': <scipy.stats....00239E53F7D90>}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [8]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
1,22,7,19,0.684532
4,22,7,18,0.684510
2,21,4,17,0.684339
9,24,8,27,0.682787
5,22,5,20,0.682313
6,19,4,22,0.682234
8,19,3,26,0.682104
0,21,6,27,0.682020
7,20,4,26,0.681918
3,21,5,25,0.681628


In [9]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 22, 'min_samples_leaf': 7, 'min_samples_split': 19}
Best CV score: 0.6845321061639283


In [10]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      7732
           1       0.97      0.98      0.97      8240

    accuracy                           0.97     15972
   macro avg       0.97      0.97      0.97     15972
weighted avg       0.97      0.97      0.97     15972



#### Evaluación

In [11]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.55      0.59      1936
           1       0.62      0.70      0.66      2060

    accuracy                           0.63      3996
   macro avg       0.63      0.62      0.62      3996
weighted avg       0.63      0.63      0.62      3996



#### Guardado

In [12]:
os.makedirs(f'./modelos_clasicos/modelos/ventanas/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/ventanas/aug_sin_proc/features_rf.pkl']